--- Day 9: Disk Fragmenter ---

Another push of the button leaves you in the familiar hallways of some friendly amphipods! Good thing you each somehow got your own personal mini submarine. The Historians jet away in search of the Chief, mostly by driving directly into walls.

While The Historians quickly figure out how to pilot these things, you notice an amphipod in the corner struggling with his computer. He's trying to make more contiguous free space by compacting all of the files, but his program isn't working; you offer to help.

He shows you the disk map (your puzzle input) he's already generated. For example:

`2333133121414131402`

The disk map uses a dense format to represent the layout of files and free space on the disk. The digits alternate between indicating the length of a file and the length of free space.

So, a disk map like 12345 would represent a one-block file, two blocks of free space, a three-block file, four blocks of free space, and then a five-block file. A disk map like 90909 would represent three nine-block files in a row (with no free space between them).

Each file on disk also has an ID number based on the order of the files as they appear before they are rearranged, starting with ID 0. So, the disk map 12345 has three files: a one-block file with ID 0, a three-block file with ID 1, and a five-block file with ID 2. Using one character for each block where digits are the file ID and . is free space, the disk map 12345 represents these individual blocks:

`0..111....22222`

The first example above, `2333133121414131402`, represents these individual blocks:

`00...111...2...333.44.5555.6666.777.888899`

The amphipod would like to move file blocks one at a time from the end of the disk to the leftmost free space block (until there are no gaps remaining between file blocks). For the disk map 12345, the process looks like this:

```
0..111....22222
02.111....2222.
022111....222..
0221112...22...
02211122..2....
022111222......
```

The first example requires a few more steps:

```
00...111...2...333.44.5555.6666.777.888899
009..111...2...333.44.5555.6666.777.88889.
0099.111...2...333.44.5555.6666.777.8888..
00998111...2...333.44.5555.6666.777.888...
009981118..2...333.44.5555.6666.777.88....
0099811188.2...333.44.5555.6666.777.8.....
009981118882...333.44.5555.6666.777.......
0099811188827..333.44.5555.6666.77........
00998111888277.333.44.5555.6666.7.........
009981118882777333.44.5555.6666...........
009981118882777333644.5555.666............
00998111888277733364465555.66.............
0099811188827773336446555566..............
```

The final step of this file-compacting process is to update the filesystem checksum. To calculate the checksum, add up the result of multiplying each of these blocks' position with the file ID number it contains. The leftmost block is in position 0. If a block contains free space, skip it instead.

Continuing the first example, the first few blocks' position multiplied by its file ID number are `0 * 0 = 0`, `1 * 0 = 0`, `2 * 9 = 18`, `3 * 9 = 27`, `4 * 8 = 32`, and so on. In this example, the checksum is the sum of these, `1928`.

Compact the amphipod's hard drive using the process he requested. **What is the resulting filesystem checksum?** (Be careful copy/pasting the input for this puzzle; it is a single, very long line.)

In [ ]:
# @author: kaenkai

import re
from random import randint


def generate_disk(n: int) -> str:
    """ Generates random disk
    Args:
        n (int): disk size
    Returns:
        (str): generated random disk
    """
    return ''.join([str(randint(0, 9)) for i in range(n)])


def read_disk(inputFile: str) -> str:
    """ Reads disk from file
    Args:
        inputFile (str): input file
    Returns:
        disk (str)
    Raises:
        FileNotFoundError if file not found
    """
    try:
        with open(inputFile, 'r') as fin:
            disk = fin.read()
        print(disk)
        return disk
    except FileNotFoundError as err:
        print(err)
        print('Using test data.')
        return '2333133121414131402'


def blocks(disk: str) -> list[str]:
    """ Converts disk to block representation, f.e.:
    2333133121414131402 -->
    00...111...2...333.44.5555.6666.777.888899

    Args:
        disk (str): disk in number representation
    Returns:
        disk (list[str]): disk in block representation
    """
    disk_blocks = []
    for i, size in enumerate(disk):
        if i%2 == 0:
            disk_blocks += [str(i//2)]*int(size)
        else: 
            disk_blocks += ['.']*int(size)
    return disk_blocks


def main(disk: list[str]) -> None:
    """Main function

    Args:
        disk (str): input disk
        files (list[int]): files ids
    """
    files = [(f, i) for i, f in enumerate(disk) if f != '.']
    free_space = disk.count('.')
    n = len(disk)
    # print(''.join(disk))
    for i, j in enumerate(disk):
        if j == '.':
            if n-i == free_space:
                break
            f, fi = files.pop()
            disk[i] = f
            disk[fi] = '.'
            # print(''.join(disk), f)
    checksum = 0
    for i, file_id in enumerate(disk[:-free_space]):
        checksum += i*int(file_id)
    print('Compacted disk:', ''.join(disk))
    print('Checksum:', checksum)


def test1():
    disk = blocks(read_disk(''))
    assert ''.join(disk) == '00...111...2...333.44.5555.6666.777.888899', disk
    main(disk)


def test2():
    rand_disk = generate_disk(21)
    print(rand_disk)
    disk = blocks(rand_disk)
    main(disk)


if __name__ == '__main__':
    # test1()
    disk = blocks(read_disk('input_day9.txt'))
    main(disk)
    # 6340197768906

--- Part Two ---

Upon completion, two things immediately become clear. First, the disk definitely has a lot more contiguous free space, just like the amphipod hoped. Second, the computer is running much more slowly! Maybe introducing all of that file system fragmentation was a bad idea?

The eager amphipod already has a new plan: rather than move individual blocks, he'd like to try compacting the files on his disk by moving whole files instead.

This time, attempt to move whole files to the leftmost span of free space blocks that could fit the file. Attempt to move each file exactly once in order of decreasing file ID number starting with the file with the highest file ID number. If there is no span of free space to the left of a file that is large enough to fit the file, the file does not move.

The first example from above now proceeds differently:

```
00...111...2...333.44.5555.6666.777.888899
0099.111...2...333.44.5555.6666.777.8888..
0099.1117772...333.44.5555.6666.....8888..
0099.111777244.333....5555.6666.....8888..
00992111777.44.333....5555.6666.....8888..
```

The process of updating the filesystem checksum is the same; now, this example's checksum would be `2858`.

Start over, now compacting the amphipod's hard drive using this new method instead. What is the resulting filesystem checksum?

In [ ]:
# @author: kaenkai

import re
from random import randint


def generate_disk(n: int) -> str:
    """ Generates random disk
    Args:
        n (int): disk size
    Returns:
        (str): generated random disk
    """
    return ''.join([str(randint(0, 9)) for i in range(n)])


def read_disk(inputFile: str) -> str:
    """ Reads disk from file
    Args:
        inputFile (str): input file
    Returns:
        disk (str)
    Raises:
        FileNotFoundError if file not found
    """
    try:
        with open(inputFile, 'r') as fin:
            disk = fin.read()
        return disk
    except FileNotFoundError as err:
        print(err)
        print('Using test data.')
        return '2333133121414131402'


def blocks(disk: str) -> list[list[str, int]]:
    """ Converts disk to block representation, f.e.:
    2333133121414131402 -->
    00...111...2...333.44.5555.6666.777.888899

    Args:
        disk (str): disk in number representation
    Returns:
        disk (list[str]): disk in block representation
    """
    disk_blocks = []
    for i, size in enumerate(disk):
        if i%2 == 0:
            disk_blocks += [[str(i//2),int(size)]]
        else: 
            disk_blocks += [['.',int(size)]]
    return disk_blocks


def conv(disk: list[list[str, int]]) -> str:
    return ''.join([f*s for f, s in disk])


def compress(disk):
    for i in range(len(disk)-1, 0, -1):
        if disk[i][0] != '.':
            for j in range(i):
                if disk[j][0] == '.' and disk[j][1] > disk[i][1]:
                    disk[i][0], disk[j][0] = disk[j][0], disk[i][0]
                    disk.insert(j+1, ['.', disk[j][1] - disk[i][1]])
                    disk[j][1] = disk[i+1][1]
                    # print(conv(disk))
                    break
                if disk[j][0] == '.' and disk[j][1] == disk[i][1]:
                    disk[i][0], disk[j][0] = disk[j][0], disk[i][0]
                    # print(conv(disk))
                    break
    return True


def main(disk: list[str]) -> None:
    """Main function

    Args:
        disk (str): input disk
        files (list[int]): files ids
    """
    # print(conv(disk))
    for _ in range(10_000):
        if compress(disk):
            break
    print('Finished after:', _, 'iterations')
    checksum = 0
    for i, b in enumerate(conv(disk)):
        if b.isnumeric():
            checksum += i*int(b)
    # print('Compacted disk:', conv(disk))
    print('Checksum:', checksum)


def test1():
    disk = blocks(read_disk(''))
    main(disk)
    assert conv(disk) == '00992111777.44.333....5555.6666.....8888..', conv(disk)


def test2():
    rand_disk = generate_disk(21)
    print(rand_disk)
    disk = blocks(rand_disk)
    main(disk)


if __name__ == '__main__':
    # test2()
    disk = blocks(read_disk('input_day9.txt'))
    main(disk)
    # 89914538143 <-- too low
    # 85481464189
    # 115886589042